<a href="https://colab.research.google.com/github/ggladish1/7130_Project/blob/main/INSY_7130_Project_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import requests

dcs_data_url = "https://github.com/ggladish1/7130_Project/raw/refs/heads/main/OCI%20Data.csv"
add_data_url = "https://github.com/ggladish1/7130_Project/raw/refs/heads/main/pivoted_data_streaming(in).csv"

dcs_data = pd.read_csv(dcs_data_url)
add_data = pd.read_csv(add_data_url)
dcs_data["Date/Time"] = pd.to_datetime(dcs_data["Date/Time"])
add_data["Timestamp"] = pd.to_datetime(add_data["Timestamp"])
dcs_data = dcs_data.set_index("Date/Time").sort_index(ascending=True)
add_data = add_data.set_index("Timestamp").sort_index(ascending=True)
dcs_data = dcs_data[~dcs_data.index.duplicated(keep="first")]
add_data = add_data[~add_data.index.duplicated(keep="first")]
data = dcs_data.join(add_data)
#data.dtypes

###Weather Data
An observed pattern is that the process becomes difficult to control during hot weather. Here, we'll use the Open-Meteo Historical Weather API to create temperature and humidity columns.

In [3]:
lat = 34.647531
long = -87.071318
# Define the date range from the existing DataFrame
start_date = data.index.min().strftime("%Y-%m-%d")
end_date = data.index.max().strftime("%Y-%m-%d")

# Request historical weather data
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": lat,
    "longitude": long,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": "temperature_2m,relative_humidity_2m",
    "temperature_unit": "fahrenheit",
    "timezone": "America/Chicago"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

# Convert API response to a DataFrame
weather = pd.DataFrame(response.json()["hourly"])

weather["time"] = pd.to_datetime(weather["time"])
weather = weather.set_index("time").sort_index()

weather = weather.rename(columns={
    "temperature_2m": "Outside Temp (F)",
    "relative_humidity_2m": "Outside RH (%)"
})

# Align hourly weather with existing measurement timestamps
weather_aligned = weather.reindex(data.index, method="nearest",
                                  tolerance=pd.Timedelta("30min"))

# Append weather variables to existing data
data = data.join(weather_aligned)

print(data[["Outside Temp (F)", "Outside RH (%)"]].head())


                     Outside Temp (F)  Outside RH (%)
Date/Time                                            
2024-10-01 02:00:00              66.2              95
2024-10-01 05:00:00              65.8              96
2024-10-01 08:00:00              70.4              89
2024-10-01 11:00:00              76.9              73
2024-10-01 14:00:00              80.9              59


In [5]:

import os
import subprocess
from pathlib import Path
from google.colab import userdata

# Replace these with your GitHub information
owner = "ggladish1"
repo = "7130_Project"

token = userdata.get("GitHub")
repo_dir = Path("/content") / repo
repo_url = f"https://github.com/{owner}/{repo}.git"

# Authentication for Git
os.environ["GITHUB_TOKEN"] = token
os.environ["GIT_ASKPASS"] = "/content/git_askpass.sh"
os.environ["GIT_TERMINAL_PROMPT"] = "0"

Path("/content/git_askpass.sh").write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) printf "%s" "$GITHUB_TOKEN" ;;\n'
    'esac\n'
)
os.chmod("/content/git_askpass.sh", 0o700)

# Clone repository if necessary
if not (repo_dir / ".git").exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True
    )


In [6]:
data.columns

Index(['SAD A MXR', 'SAD C MXR', 'RxA %AO', 'RxA MO', 'RxB %AO', 'RxC %AO',
       'RxC MO', 'RxD %AO', 'DWPC AO', 'DWPC PSD', 'DWPC MO', 'CF AO',
       'CF PSD', 'COP AO', 'COP MO', 'COP PSD', '1178 AO', 'T1178 PSD',
       'A_MSL_AMSL_FDRATE_MEAS', 'A_MSL_AMSL_SETPT_OUT',
       'C_MSL_CMSL_FDRATE_MEAS', 'C_MSL_CMSL_SETPT_OUT', 'Outside Temp (F)',
       'Outside RH (%)'],
      dtype='object')

In [7]:

output_dir = repo_dir / "output"
output_dir.mkdir(exist_ok=True)

data.to_csv(
    output_dir / "process_data_complete.csv",
    index_label="Date/Time"
)

weather_aligned.to_csv(
    output_dir / "weather_aligned.csv",
    index_label="Date/Time"
)


In [8]:

subprocess.run(
    ["git", "config", "user.name", "ggladish1"],
    cwd=repo_dir, check=True
)
subprocess.run(
    ["git", "config", "user.email", "ggladish1@gmail.com"],
    cwd=repo_dir, check=True
)

subprocess.run(
    ["git", "add", "output/process_data_complete.csv",
     "output/weather_aligned.csv"],
    cwd=repo_dir, check=True
)

subprocess.run(
    ["git", "commit", "-m",
     "Add prepared process data and aligned weather data"],
    cwd=repo_dir, check=True
)

subprocess.run(
    ["git", "push"],
    cwd=repo_dir, check=True
)


CompletedProcess(args=['git', 'push'], returncode=0)